# 06 - Next Trading Day Signal and Reporting Engine

Goal: generate next-day buy/sell/hold indications, inspect history with candlestick charts, and export an investment impact report.

The dataset is daily historical market data, so "24 hrs" means the next available trading day after the latest row in the dataset.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(ROOT / 'src'))

import joblib
import pandas as pd
import plotly.express as px

from invest_intel.data import build_returns_matrix, load_prices, load_processed
from invest_intel.features import drop_modeling_na, modeling_feature_columns
from invest_intel.modeling import train_direction_model, train_return_models
from invest_intel.portfolio import construct_profile_portfolio
from invest_intel.reporting import build_investment_report, save_investment_report
from invest_intel.risk import risk_summary
from invest_intel.signals import signals_from_saved_model
from invest_intel.visualization import candlestick_with_volume

## Train One-Day Return and Direction Models

In [2]:
features = load_processed('nifty50_features.csv')
feature_cols = modeling_feature_columns(features)

return_df = drop_modeling_na(features, 'future_return_1d', feature_cols)
return_results = train_return_models(return_df, feature_cols, target_col='future_return_1d', test_start_date='2019-01-01')
metrics = pd.DataFrame([{'model': result.name, **result.metrics} for result in return_results])
metrics

,model,mae,rmse,r2,directional_accuracy
0,ridge_regression,0.016863,0.026298,-0.012240,0.512188
1,hist_gradient_boosting,0.017205,0.026829,-0.053532,0.501098


In [3]:
best = max(return_results, key=lambda result: result.metrics['directional_accuracy'])
(ROOT / 'models').mkdir(exist_ok=True)
joblib.dump({'model': best.estimator, 'feature_cols': feature_cols, 'target': 'future_return_1d'}, ROOT / 'models' / 'stock_return_1d_model.joblib')

direction_df = drop_modeling_na(features, 'target_direction_1d', feature_cols)
direction_result = train_direction_model(direction_df, feature_cols, target_col='target_direction_1d')
joblib.dump({'model': direction_result.estimator, 'feature_cols': feature_cols, 'target': 'target_direction_1d'}, ROOT / 'models' / 'stock_direction_1d_model.joblib')
direction_result.metrics

{'directional_accuracy': 0.5137117346938775,
 'balanced_accuracy': 0.5129968151612441,
 'precision': 0.5182677567705672,
 'recall': 0.5675316499965027,
 'f1': 0.5417821253296832,
 'roc_auc': 0.5204741402852098,
 'decision_threshold': 0.49,
 'validation_directional_accuracy': 0.5158488927485888}

## Generate Buy/Sell/Hold Signals

In [4]:
signals = signals_from_saved_model(features)
signals.head(25)

,date,symbol,close,predicted_return_1d,predicted_price_1d,technical_score,signal,confidence,signal_reason,signal_source
36,2021-04-30,ONGC,108.15,0.006429,108.845282,2.3,BUY,0.790520,close above 20-day MA; close above 50-day MA; ...,model_plus_indicators
46,2021-04-30,TCS,3035.65,-0.001878,3029.949199,-2.8,SELL,0.746603,close below 20-day MA; close below 50-day MA; ...,model_plus_indicators
15,2021-04-30,HCLTECH,898.95,-0.001766,897.362174,-2.8,SELL,0.744594,close below 20-day MA; close below 50-day MA; ...,model_plus_indicators
49,2021-04-30,ULTRACEMCO,6278.95,-0.001593,6268.944810,-2.8,SELL,0.741482,close below 20-day MA; close below 50-day MA; ...,model_plus_indicators
32,2021-04-30,MARUTI,6455.65,-0.001470,6446.162871,-2.8,SELL,0.739253,close below 20-day MA; close below 50-day MA; ...,model_plus_indicators
18,2011-08-05,HEROHONDA,1785.35,-0.000995,1783.572809,-2.8,SELL,0.730718,close below 20-day MA; close below 50-day MA; ...,model_plus_indicators
33,2012-01-16,MUNDRAPORT,135.50,0.000553,135.574905,2.8,BUY,0.722751,close above 20-day MA; close above 50-day MA; ...,model_plus_indicators
44,2021-04-30,TATAMOTORS,293.85,-0.000510,293.700208,-2.8,SELL,0.721976,close below 20-day MA; close below 50-day MA; ...,model_plus_indicators
41,2021-04-30,SHREECEM,27910.50,-0.000263,27903.146572,-2.8,SELL,0.717542,close below 20-day MA; close below 50-day MA; ...,model_plus_indicators
47,2021-04-30,TECHM,960.40,-0.000172,960.234353,-2.8,SELL,0.715905,close below 20-day MA; close below 50-day MA; ...,model_plus_indicators


In [5]:
px.bar(
    signals.head(25),
    x='symbol',
    y='predicted_return_1d',
    color='signal',
    hover_data=['confidence', 'signal_reason'],
    title='Next trading day signal leaderboard',
)

## Candlestick History

In [6]:
prices = load_prices()
symbol = 'RELIANCE'
recent_prices = prices[(prices['symbol'] == symbol) & (prices['date'] >= '2020-01-01')]
candlestick_with_volume(recent_prices, symbol, title=f'{symbol} recent candlestick history')

## Portfolio Impact Report

In [7]:
returns = build_returns_matrix(prices[prices['date'] >= '2015-01-01'], min_obs=252).dropna(how='all')
risk = risk_summary(returns)
weights, metrics, selected = construct_profile_portfolio(returns, profile='balanced')
report = build_investment_report(signals, risk, selected)
path = save_investment_report(report)
display(report.head(25))
path

,date,symbol,signal,confidence,predicted_return_1d,predicted_price_1d,close,weight,portfolio_impact_score,annualized_return,annualized_volatility,sharpe_ratio,sortino_ratio,max_drawdown,var_95,expected_shortfall_95,signal_reason,signal_source
44,2021-04-30,BAJAJFINSV,HOLD,0.597615,-0.004288,10994.302614,11041.65,0.148977,-0.000382,0.413608,0.364003,0.971442,1.326201,-0.585896,-0.031063,-0.049115,close above 20-day MA; close above 50-day MA; ...,model_plus_indicators
37,2021-04-30,TITAN,HOLD,0.619573,-0.000732,1490.558351,1491.65,0.099839,-0.000045,0.246423,0.328475,0.567540,0.851798,-0.417300,-0.028412,-0.043129,close below 20-day MA; close above 50-day MA; ...,model_plus_indicators
33,2021-04-30,ASIANPAINT,SELL,0.630644,-0.001347,2532.983723,2536.40,0.099110,-0.000084,0.216985,0.265990,0.590192,0.880887,-0.288550,-0.023899,-0.035366,close below 20-day MA; close above 50-day MA; ...,model_plus_indicators
29,2021-04-30,HINDUNILVR,SELL,0.642978,-0.002032,2348.966875,2353.75,0.097076,-0.000127,0.200041,0.237602,0.589394,0.995604,-0.213217,-0.020818,-0.029141,close below 20-day MA; close above 50-day MA; ...,model_plus_indicators
22,2021-04-30,BAJFINANCE,HOLD,0.671087,-0.008949,5403.109296,5451.90,0.089725,-0.000539,0.073988,0.540944,0.025859,0.023358,-0.932820,-0.036420,-0.069521,close above 20-day MA; close above 50-day MA; ...,model_plus_indicators
8,2021-04-30,SHREECEM,SELL,0.717542,-0.000263,27903.146572,27910.50,0.080609,-0.000015,0.191990,0.312928,0.421792,0.673155,-0.368969,-0.028938,-0.041025,close below 20-day MA; close below 50-day MA; ...,model_plus_indicators
18,2021-04-30,NESTLEIND,SELL,0.693610,0.001066,16326.637262,16309.25,0.075210,0.000056,0.164980,0.255114,0.411501,0.654740,-0.324011,-0.021323,-0.032935,close below 20-day MA; close below 50-day MA; ...,model_plus_indicators
27,2021-04-30,TATASTEEL,SELL,0.648880,-0.007916,1025.815291,1034.00,0.065741,-0.000338,0.163166,0.390693,0.264059,0.394455,-0.676133,-0.037441,-0.054760,close above 20-day MA; close above 50-day MA; ...,model_plus_indicators
3,2021-04-30,ULTRACEMCO,SELL,0.741482,-0.001593,6268.944810,6278.95,0.063437,-0.000075,0.148144,0.292208,0.301650,0.449343,-0.376561,-0.027575,-0.038734,close below 20-day MA; close below 50-day MA; ...,model_plus_indicators
39,2021-04-30,RELIANCE,SELL,0.606080,-0.005804,1982.923023,1994.50,0.062210,-0.000219,0.139186,0.363351,0.217932,0.226786,-0.526756,-0.025607,-0.045384,close above 20-day MA; close below 50-day MA; ...,model_plus_indicators


WindowsPath('E:/Projects/cult_invest_intel_ml/reports/investment_signal_report.csv')